# Musterlösungen Woche 9: Sichtbarkeit & Information Hiding

Dieses Notebook enthält die vollständigen Musterlösungen sowie kurze Begründungen zu allen Aufgaben.

---
## Teil 2: Sichtbarkeitsattribute setzen (Lösungen)

### Aufgabe 2.1: Klassen und Methoden über zwei Packages

**Begründung der Sichtbarkeiten:**
- `public class A1`, `public class A2`, `public class B1`: Alle drei Klassen werden jeweils aus dem anderen Package referenziert (`B1` erbt von `A1` und ruft `A2.a22()` auf; `A2` ruft `B1.b12()` auf) und müssen daher `public` sein.
- `A1.a11()`: Wird in `A2.a21()` (gleiches Package `A`) und in `B1.b11()` (Unterklasse in anderem Package `B`) aufgerufen → **`protected`**.
- `A1.a12()`: Wird ausschliesslich innerhalb von `A1` (`a11()`) aufgerufen → **`private`**.
- `A2.a21()`: Wird in `A1.a12()` (gleiches Package `A`, keine Vererbung) aufgerufen → ***package-private* (kein Modifikator)**.
- `A2.a22()`: Wird in `B1.b12()` (anderes Package `B`, keine Vererbung zwischen `B1` und `A2`) aufgerufen → **`public`**.
- `B1.b11()`: Wird ausschliesslich innerhalb von `B1` (`b12()`) aufgerufen → **`private`**.
- `B1.b12()`: Wird in `A2.a22()` (anderes Package `A`, keine Vererbung) aufgerufen → **`public`**.

In [ ]:
package A;

import B.B1;

public class A1 {

    protected static void a11() {
        System.out.println("a11");
        A1.a12();
    }

    private static void a12() {
        System.out.println("a12");
        A2.a21();
    }
}

public class A2 {

    static void a21() { // package-private
        System.out.println("a21");
        A1.a11();
    }

    public static void a22() {
        System.out.println("a22");
        B1.b12();
    }
}

In [ ]:
package B;

import A.A1;
import A.A2;

public class B1 extends A1 {

    private static void b11() {
        System.out.println("b11");
        A1.a11();
    }

    public static void b12() {
        System.out.println("b12");
        A2.a22();
        B1.b11();
    }
}

### Aufgabe 2.2: Klassen, Felder, Konstruktoren und Methoden über zwei Packages

**Begründung der Sichtbarkeiten:**
- `public class Buch` und `public class VerlagsKatalog`: Werden aus dem Package `shop` heraus verwendet → **`public`**.
- `class SonderEdition` und `class ShopApp`: Werden nur innerhalb des eigenen Packages `shop` verwendet → ***package-private***.
- `Buch.titel`: Wird in `Buch` und in `VerlagsKatalog` (gleiches Package `verlag`) gelesen → ***package-private***.
- `Buch.interneNotiz` und `SonderEdition.rabatt`: Werden nur in der eigenen Klasse verwendet → **`private`**.
- `Buch.basisPreis`: Wird in `Buch` und in der Unterklasse `SonderEdition` (anderes Package `shop`) gelesen → **`protected`**.
- Konstruktor `Buch(...)`: Wird nur über `super(...)` in der Unterklasse `SonderEdition` (anderes Package `shop`) aufgerufen → **`protected`**.
- Konstruktor `SonderEdition(...)`: Wird nur in `ShopApp` (gleiches Package `shop`) aufgerufen → ***package-private***.
- `Buch.getInterneNotiz()`: Wird nur innerhalb von `Buch` aufgerufen → **`private`**.
- `Buch.formatiereInfo()`: Wird in `VerlagsKatalog` (gleiches Package `verlag`) aufgerufen → ***package-private***.
- `VerlagsKatalog.druckeKatalogEintrag(...)`: Wird aus `ShopApp` (anderes Package `shop`) aufgerufen → **`public`**.
- `SonderEdition.berechneEndpreis()`: Wird nur in `ShopApp` (gleiches Package `shop`) aufgerufen → ***package-private***.

In [ ]:
package verlag;

public class Buch {
    String titel;                  // package-private
    private String interneNotiz;   // private
    protected double basisPreis;   // protected

    protected Buch(String titel, double basisPreis, String interneNotiz) {
        this.titel = titel;
        this.basisPreis = basisPreis;
        this.interneNotiz = interneNotiz;
    }

    private String getInterneNotiz() {
        return this.interneNotiz;
    }

    String formatiereInfo() {      // package-private
        return this.titel + " (" + this.getInterneNotiz() + ")";
    }
}

public class VerlagsKatalog {
    public static void druckeKatalogEintrag(Buch buch) {
        System.out.println(buch.titel + ": " + buch.formatiereInfo());
    }
}

In [ ]:
package shop;

import verlag.Buch;
import verlag.VerlagsKatalog;

class SonderEdition extends Buch { // package-private
    private double rabatt;

    SonderEdition(String titel, double basisPreis, double rabatt) { // package-private
        super(titel, basisPreis, "Limitierte Auflage");
        this.rabatt = rabatt;
    }

    double berechneEndpreis() {    // package-private
        return this.basisPreis * (1.0 - this.rabatt);
    }
}

class ShopApp {                    // package-private
    public static void main(String[] args) {
        SonderEdition edition = new SonderEdition("Java Kompakt", 40.0, 0.25);
        VerlagsKatalog.druckeKatalogEintrag(edition);
        System.out.println("Endpreis: " + edition.berechneEndpreis() + " CHF");
    }
}

ShopApp.main(new String[0]);

---
## Teil 3: Information Hiding (Lösungen)

### Aufgabe 3.1: Geschütztes Buch (`ProtectedBook`)

**Lösung:**
- Der Konstruktor `ProtectedBook(String title)` wird **`private`** gemacht, sodass Objekte nur noch kontrolliert über die Factory-Methode `getBook(...)` (mit Passwortprüfung) erzeugt werden können.
- Zusätzlich wird das Attribut `title` als **`private`** deklariert, damit es nicht von aussen manipuliert werden kann.
- `EagerReader` scheitert nun beim Kompilieren an `new ProtectedBook("Illegal Lord of the Rings")` (`ProtectedBook(String) has private access in ProtectedBook`).

In [ ]:
public class ProtectedBook {

    private String title;

    private ProtectedBook(String title) {
        this.title = title;
        if (title != null) {
            System.out.println("Created the following Book: " + title);
        }
    }

    private static ProtectedBook createEmptyBook() {
        return new ProtectedBook(null);
    }

    public static ProtectedBook getBook(String title, String password) {
        if ("reader".equals(password)) {
            return new ProtectedBook(title);
        }
        else {
            System.out.println("Wrong password! Returned empty book.");
            return createEmptyBook();
        }
    }

}

public class Main {

    public static void main(String[] args) {
        ProtectedBook.getBook("Lord of the Rings", "readr");
        System.out.println("Oops, mistyped");
        ProtectedBook.getBook("Lord of the Rings", "reader");
    }
}

Main.main(new String[0]);

In [ ]:
// Kompiliert wie gewünscht NICHT mehr (ProtectedBook-Konstruktor ist private):
public class EagerReader {

    public static void main(String[] args) {
        ProtectedBook.getBook("Lord of the Rings", "?");
        System.out.println("I don't know the password");
        new ProtectedBook("Illegal Lord of the Rings");
    }
}

EagerReader.main(new String[0]);

### Aufgabe 3.2: Spieleinstellungen (`GameSettings`)

**Lösung:**
- Das Feld `maxAttempts` wird **`private`** gemacht, sodass `Villain` es nicht mehr direkt überschreiben kann (`GameSettings.maxAttempts = 10;`).
- Für den lesenden Zugriff in `Main.playGame()` wird eine öffentliche Getter-Methode `public static int getMaxAttempts()` bereitgestellt.
- *(Hinweis: Falls der Wert niemals geändert werden soll, könnte man alternativ auch `public static final int MAX_ATTEMPTS = 3;` verwenden; `private` + Getter ist jedoch die klassische Information-Hiding-Lösung, da die interne Speicherung verborgen bleibt.)*

In [ ]:
public class GameSettings {
    private static int maxAttempts = 3;

    public static int getMaxAttempts() {
        return maxAttempts;
    }

    public static void display() {
        System.out.println(
            "Maximum attempts: " + maxAttempts
        );
    }
}

public class Main {

    public static void playGame() {
        for (int i = 0; i < GameSettings.getMaxAttempts(); i++) {
            System.out.println("Playing...");
        }
    }

    public static void main(String[] args) {
        GameSettings.display();
        playGame();
    }
}

Main.main(new String[0]);

In [ ]:
// Kompiliert wie gewünscht NICHT mehr (maxAttempts ist private):
public class Villain {

    public static void sabotage() {
        GameSettings.maxAttempts = 10;
        System.out.println("I want to play longer!");
    }

    public static void main(String[] args) {
        GameSettings.display();
        sabotage();
        GameSettings.display();

        Main.playGame();
    }
}

Villain.main(new String[0]);

### Aufgabe 3.3: Lagerbestand (`Inventory`)

**Lösung:**
- Obwohl `stock` bereits `private` war, gab `public int[] getStock()` eine **Referenz auf das interne Array** nach aussen. Dadurch konnte `Villain` die Array-Elemente direkt auf `0` setzen.
- Wir entfernen `public int[] getStock()` (bzw. machen es `private`) und bieten stattdessen einen kontrollierten Lesezugriff auf einzelne Elemente an: `public int getStockAt(int index)`.
- `Main.totalAmount(...)` nutzt nun `inventory.getStockAt(i)`, während `Villain` nicht mehr kompiliert, da `getStock()` nicht mehr öffentlich existiert.
- *(Hinweis: Müsste man zwingend ein ganzes Array zurückgeben, würde man eine **defensive Kopie** via `stock.clone()` zurückgeben – dann würde `Villain` allerdings noch kompilieren und nur zur Laufzeit wirkungslos bleiben.)*

In [ ]:
public class Inventory {
    private int[] stock;

    public Inventory() {
        stock = new int[]{10, 20, 30};
    }

    public int getStockAt(int index) {
        return stock[index];
    }

    public int stockSize() {
        return stock.length;
    }

    public void display() {
        for (int amount : stock) {
            System.out.println(amount);
        }
    }
}

public class Main {

    public static int totalAmount(Inventory inventory) {
        int n = inventory.stockSize();
        int total = 0;
        for (int i = 0; i < n; i++) {
            total += inventory.getStockAt(i);
        }
        return total;
    }

    public static void main(String[] args) {
        Inventory inventory = new Inventory();

        inventory.display();
        System.out.println("Total:");
        System.out.println(totalAmount(inventory));
    }
}

Main.main(new String[0]);

In [ ]:
// Kompiliert wie gewünscht NICHT mehr (getStock() existiert nicht mehr öffentlich):
public class Villain {

    public static void sabotage(Inventory inventory) {
        int[] stock = inventory.getStock();

        stock[0] = 0;
        stock[1] = 0;
        stock[2] = 0;

        System.out.println("Muahahaha!");
    }

    public static void main(String[] args) {
        Inventory inventory = new Inventory();

        inventory.display();
        sabotage(inventory);
        inventory.display();
    }
}

Villain.main(new String[0]);

### Aufgabe 3.4: Zähler (`Counter`)

**Lösung:**
- Das Feld `count` wird **`private`** gemacht, sodass niemand den Zählerstand direkt manipulieren oder auf `0` zurücksetzen kann (`counter.count = 0;`).
- Statt eines allgemeinen Setters stellen wir eine fachlich sinnvolle Methode `public void increment()` (sowie optional `public int getCount()`) bereit, mit der der Zähler nur kontrolliert um `1` erhöht werden kann.
- `Main.countEven(...)` ruft `counter.increment()` auf, während `Villain` beim Kompilieren fehlschlägt.

In [ ]:
public class Counter {
    private int count;

    public Counter() {
        this.count = 0;
    }

    public void increment() {
        this.count++;
    }

    public int getCount() {
        return this.count;
    }

    public void display() {
        System.out.println(this.count);
    }

}

public class Main {

    public static void countEven(int[] arr, Counter counter) {
        for (int i : arr) {
            if (i % 2 == 0) {
                counter.increment();
            }
        }
    }

    public static void main(String[] args) {
        Counter evenCounter = new Counter();
        int[] arr = new int[]{1,2,5,8,3,6};
        countEven(arr, evenCounter);
        evenCounter.display();

    }
}

Main.main(new String[0]);

In [ ]:
// Kompiliert wie gewünscht NICHT mehr (count ist private):
public class Villain {

    public static void stealCount(Counter counter) {
        counter.count = 0;
        System.out.println("Muahahaha!");
    }

    public static void main(String[] args) {
        Counter evenCounter = new Counter();
        int[] arr = new int[]{1,2,5,8,3,6};
        Main.countEven(arr, evenCounter);
        evenCounter.display();
        stealCount(evenCounter);
        evenCounter.display();

    }

}

Villain.main(new String[0]);